# Inclusive jet RAA in 0–10% Pb–Pb at 5.02 TeV

Run this notebook with a kernel from the environment where you installed the published wheels.

```bash
pip install 'hepyy[hepbundle,jewel-cpp,glauber-cpp]>=0.2.36' hepyy-utils matplotlib jupyterlab
pip install 'hepyy[cppyy]'  # If cppyy is missing
```

The wheels provide the event runner and analysis scripts. For the PDF grid and macOS cppyy setup, see the [README](https://github.com/matplo/jewel-cpp#inclusive-jet-r_aa). The full run generates 60,000 Glauber proposals, then 3,500 hard events each for pp and zero-interaction Pb–Pb and 3,500 for quenched Pb–Pb. Allow several minutes and up to 1 GB for outputs.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import sysconfig
import tempfile
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
from hepyy_jewel_cpp.cli import ANALYSIS_TOOLS, NATIVE_RUNNER

scripts = Path(sysconfig.get_path("scripts"))
work = Path(tempfile.mkdtemp(prefix="jet-raa-", dir=Path.cwd()))
env = os.environ.copy()

def run(label, *args):
    log = work / f"{label}.log"
    with log.open("w") as stream:
        result = subprocess.run(args, cwd=work, env=env, stdout=stream,
                                stderr=subprocess.STDOUT)
    if result.returncode:
        print("\n".join(log.read_text().splitlines()[-30:]))
        result.check_returncode()
    print(f"{label} finished. Log: {log.name}")

print(work)

## Generate the minimum-bias geometry

JEWEL samples impact parameters from the 0–10% class with probability proportional to the number of binary collisions. The Pb–Pb normalization uses the **unweighted** class mean nuclear overlap.

In [ ]:
run("glauber", scripts / "glauber-pbpb", "pbpb-5020", "60000", "72913")
geometry = work / "pbpb-5020" / "events.csv"
metadata = json.loads(geometry.with_name("metadata.json").read_text())
print(f"{metadata['inelastic_events']:,} inelastic Pb–Pb events")

## Generate matched samples

Use CT14nlo member 0 with a 20–100 GeV hard-scale window and power-5 bias. Set `pdf_dir` to an existing directory containing the CT14nlo grid to reuse it. The zero-interaction sample keeps the Pb–Pb geometry with medium interactions switched off.

In [ ]:
pdf_dir = work / "pdfsets"  # Or Path("/path/to/existing/pdfsets")
pdf_dir.mkdir(parents=True, exist_ok=True)
if not (pdf_dir / "CT14nlo" / "CT14nlo_0000.dat").is_file():
    archive = work / "CT14nlo.tar.gz"
    urlretrieve("https://lhapdfsets.web.cern.ch/current/CT14nlo.tar.gz", archive)
    subprocess.run(["tar", "-xzf", archive, "-C", pdf_dir], check=True)
env["LHAPDF_DATA_PATH"] = os.pathsep.join(
    part for part in (str(pdf_dir.resolve()), env.get("LHAPDF_DATA_PATH")) if part
)

common = (
    "--native-mpi", "--compact-hadrons", "--pthat-bias", "5",
    "--table-energy", "jewel", "--longitudinal-boost", "on",
    "--jewel-fragmentation", "--ecm", "5020", "--pt-min", "20",
    "--pt-max", "100", "--lhapdf", "CT14nlo/0",
    "--hard-alpha-s", "0.11754434294459773", "--hard-alpha-s-order", "1",
    "--isr-alpha-s", "0.11754434294459773", "--isr-alpha-s-order", "1",
    "--alpha-s-nfmax", "5", "--isr-alpha-s-cmw", "off",
)
central = ("--centrality", "0", "10", "--glauber-events", geometry)
runner = scripts / "jewel-run-events"
run("pp", runner, "pp.jsonl", "3500", "2600611", "vacuum", *common)
run("control", runner, "control.jsonl", "3500", "2600612",
    "zero-interaction", *common, *central)
run("quenched", runner, "quenched.jsonl", "3500", "2600514",
    "medium", *common, *central)

## Reconstruct jets and check closure

The bundled analyzer uses hepyy FastJet with anti-$k_t$, $R=0.4$, and $|\eta|<2$. The closure requires at least three bins with 50 or more jets in each sample. Every eligible bin must agree with unity within three standard errors.

In [ ]:
def analyze(sample, *flags):
    jets = f"{sample}-jets.json"
    result = f"{sample}-raa.json"
    run(f"{sample}-jets", sys.executable, ANALYSIS_TOOLS / "analyze_native_events.py",
        "pp.jsonl", f"{sample}.jsonl", "--generator", NATIVE_RUNNER, "--output", jets)
    run(f"{sample}-raa", sys.executable, ANALYSIS_TOOLS / "jet_raa.py",
        jets, "--output", result, *flags)
    return json.loads((work / result).read_text())

control = analyze("control", "--validate-no-quenching")
quenched = analyze("quenched")
closure = control["no_quenching_closure"]
taa = control["taa"]
print(f"Closure passed in {closure['bins_checked']} bins. "
      f"Largest deviation: {closure['max_abs_pull']:.2f} standard errors.")
print(f"Mean T_AA: {taa['value_mb_inverse']:.4f} ± "
      f"{taa['uncertainty_mb_inverse']:.4f} mb⁻¹")

## Plot the control and quenched ratios

Horizontal bars show bin widths. Vertical bars combine the event-level spectrum and generator cross-section uncertainties. The common nuclear overlap cancels in the ratio. Both curves show only bins with at least 50 jets in each sample and a defined ratio uncertainty. The full bin results are in the output JSON files.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6), layout="constrained")
ax.axhline(1, color="0.45", linestyle="--", linewidth=1)
for report, label, marker, color in (
    (control, "Zero-interaction Pb–Pb / pp", "o", "#126180"),
    (quenched, "Quenched Pb–Pb / pp", "s", "#a54526"),
):
    bins = [row for row in report["bins"]
            if row["raa_standard_error"] is not None
            and min(row["pp"]["raw_jets"], row["pbpb"]["raw_jets"]) >= 50]
    edges = [(row["pt_low_GeV"], row["pt_high_GeV"]) for row in bins]
    ax.errorbar([(lo + hi) / 2 for lo, hi in edges], [row["raa"] for row in bins],
                xerr=[(hi - lo) / 2 for lo, hi in edges],
                yerr=[row["raa_standard_error"] for row in bins],
                fmt=marker, capsize=3, color=color, label=label)
ax.set(xlabel="Jet transverse momentum (GeV)", ylabel=r"$R_{AA}$",
       title="Inclusive anti-$k_t$ jets in 0–10% Pb–Pb at 5.02 TeV")
ax.legend()
fig.savefig(work / "jet-raa.png", dpi=160)
plt.show()